In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from langdetect import detect, DetectorFactory
import seaborn as sns


In [2]:
bundle = joblib.load("multilabel_project_bundle.pkl")

model = bundle["model"]
thresholds = bundle["thresholds"]
label_cols = bundle["label_cols"]

print("✅ Model loaded")
print("Labels:", label_cols)
print("Thresholds:", thresholds)

✅ Model loaded
Labels: ['Attitude_Towards_Students', 'Campus_conditions', 'Corruption', 'Academic_Process_Management', 'Education_Quality']
Thresholds: [0.51, 0.525, 0.5, 0.47, 0.5]


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("✅ MPNet loaded")

✅ MPNet loaded


In [5]:
full_df = pd.read_csv("full_df.csv", sep = ',')
full_df .info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6254 entries, 0 to 6253
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   University Name              6254 non-null   object 
 1   Review Text                  6254 non-null   object 
 2   Cleaned_Text                 2279 non-null   object 
 3   Language                     6254 non-null   object 
 4   Year                         6254 non-null   int64  
 5   Sentiment_Category           6254 non-null   object 
 6   Attitude_Towards_Students    615 non-null    float64
 7   Campus_conditions            614 non-null    float64
 8   Corruption                   614 non-null    float64
 9   Academic_Process_Management  614 non-null    float64
 10  Education_Quality            614 non-null    float64
dtypes: float64(5), int64(1), object(5)
memory usage: 537.6+ KB


In [6]:
negative_unlabeled = full_df[
    full_df["Sentiment_Category"] == "negative"
].copy()

print(negative_unlabeled.shape)

(1676, 11)


In [12]:
texts = negative_unlabeled["Cleaned_Text"].fillna("").tolist()

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

In [13]:
probs = model.predict_proba(embeddings)
preds = (
    probs >= np.array(thresholds)
).astype(int)
for i, label in enumerate(label_cols):

    negative_unlabeled[f"{label}_prob"] = probs[:, i]
for i, label in enumerate(label_cols):

    negative_unlabeled[f"{label}_pred"] = preds[:, i]
negative_unlabeled["Predicted_Issues"] = [
    ", ".join(
        [
            label_cols[j]
            for j in range(len(label_cols))
            if row[j] == 1
        ]
    )
    for row in preds
]

negative_unlabeled.tail()

,University Name,Review Text,Cleaned_Text,Language,Year,Sentiment_Category,Attitude_Towards_Students,Campus_conditions,Corruption,Academic_Process_Management,...,Campus_conditions_prob,Corruption_prob,Academic_Process_Management_prob,Education_Quality_prob,Attitude_Towards_Students_pred,Campus_conditions_pred,Corruption_pred,Academic_Process_Management_pred,Education_Quality_pred,Predicted_Issues
2274,University_19,"Если вы надеетесь чему то научиться, получить ...","Если вы надеетесь чему то научиться, получить ...",ru,2016,negative,NaN,NaN,NaN,NaN,...,0.096939,0.448327,0.369244,0.887351,0,0,0,0,1,Education_Quality
2275,University_19,"каждый раз при открытии word документа, или да...","каждый раз при открытии word документа, или да...",ru,2016,negative,NaN,NaN,NaN,NaN,...,0.460610,0.292557,0.922218,0.451797,0,0,0,1,0,Academic_Process_Management
2276,University_19,"They will just take your money that's all, \nY...","They will just take your money that' s all, Yo...",en,2016,negative,NaN,NaN,NaN,NaN,...,0.043528,0.903791,0.211025,0.701291,0,0,1,0,1,"Corruption, Education_Quality"
2277,University_19,Не розумію чому рейтинг цього ВНЗ такий високи...,Не розумію чому рейтинг цього ВНЗ такий високи...,uk,2016,negative,NaN,NaN,NaN,NaN,...,0.356146,0.368243,0.404405,0.422597,0,0,0,0,0,
2278,University_1,"учусь 4й год,с первого курса обещают дать акре...","учусь 4й год, с первого курса обещают дать акр...",ru,2010,negative,NaN,NaN,NaN,NaN,...,0.195268,0.397758,0.695162,0.691268,0,0,0,1,1,"Academic_Process_Management, Education_Quality"


In [14]:
# # Update full dataset with processed negative unlabeled records
full_df.loc[
    negative_unlabeled.index,
    negative_unlabeled.columns
] = negative_unlabeled

In [22]:
full_df[full_df["Sentiment_Category"] == 'negative'].tail()

,University Name,Review Text,Cleaned_Text,Language,Year,Sentiment_Category,Attitude_Towards_Students,Campus_conditions,Corruption,Academic_Process_Management,...,Campus_conditions_prob,Corruption_prob,Academic_Process_Management_prob,Education_Quality_prob,Attitude_Towards_Students_pred,Campus_conditions_pred,Corruption_pred,Academic_Process_Management_pred,Education_Quality_pred,Predicted_Issues
2274,University_19,"Если вы надеетесь чему то научиться, получить ...","Если вы надеетесь чему то научиться, получить ...",ru,2016,negative,NaN,NaN,NaN,NaN,...,0.096939,0.448327,0.369244,0.887351,0.0,0.0,0.0,0.0,1.0,Education_Quality
2275,University_19,"каждый раз при открытии word документа, или да...","каждый раз при открытии word документа, или да...",ru,2016,negative,NaN,NaN,NaN,NaN,...,0.460610,0.292557,0.922218,0.451797,0.0,0.0,0.0,1.0,0.0,Academic_Process_Management
2276,University_19,"They will just take your money that's all, \nY...","They will just take your money that' s all, Yo...",en,2016,negative,NaN,NaN,NaN,NaN,...,0.043528,0.903791,0.211025,0.701291,0.0,0.0,1.0,0.0,1.0,"Corruption, Education_Quality"
2277,University_19,Не розумію чому рейтинг цього ВНЗ такий високи...,Не розумію чому рейтинг цього ВНЗ такий високи...,uk,2016,negative,NaN,NaN,NaN,NaN,...,0.356146,0.368243,0.404405,0.422597,0.0,0.0,0.0,0.0,0.0,
2278,University_1,"учусь 4й год,с первого курса обещают дать акре...","учусь 4й год, с первого курса обещают дать акр...",ru,2010,negative,NaN,NaN,NaN,NaN,...,0.195268,0.397758,0.695162,0.691268,0.0,0.0,0.0,1.0,1.0,"Academic_Process_Management, Education_Quality"


In [24]:
# frequency of problems
label_cols_pred = [
    "Attitude_Towards_Students_pred",
    "Campus_conditions_pred",
    "Corruption_pred",
    "Academic_Process_Management_pred",
    "Education_Quality_pred"
]

issue_counts = full_df[label_cols_pred].sum().sort_values(ascending=False)

issue_counts

Education_Quality_pred              819.0
Academic_Process_Management_pred    658.0
Attitude_Towards_Students_pred      572.0
Corruption_pred                     570.0
Campus_conditions_pred              317.0
dtype: float64

In [25]:
full_df.to_excel("student_feedback.xlsx")

### Qualitative Analysis of Predictions

**1. Highest-confidence predictions**

The examples below correspond to the highest-confidence predictions produced by the model. They are intended to illustrate the semantic patterns learned by the classifier rather than to represent its overall performance. Model quality is evaluated separately using the validation and test metrics reported above.

In [69]:
for label in label_cols:

    print(f"\n{'='*100}")
    print(f"🏷️ {label}")
    print(f"{'='*100}")

    top_reviews = negative_unlabeled.sort_values(
        f"{label}_prob",
        ascending=False
    ).head(5)

    for i, (_, row) in enumerate(top_reviews.iterrows(), 1):

        print(f"\n🔹 Example #{i}")
        print(f"📈 Probability: {row[f'{label}_prob']:.3f}")
        print(f"📝 {row['Review Text']}")


🏷️ Attitude_Towards_Students

🔹 Example #1
📈 Probability: 0.969
📝 ненавиджу цей університет.викладачі постійно повторюють,що ми,студенти,дорослі люди,ми їх колеги і т.д.,а самі ставляться як до бидлоти до "колег"!У розмові з викладачем повинен підтакувати,погоджуватись у всьому,завжди (незалежно від ситуації) у відносинах викладач-студент,студент - ніхто і ніщо.Коли ти виправдовуєшся,відстоюєш свою точку зору...викладач каже:"Не хаміть!"(!)...про що може йти річ?..Після такого навіть привітатись боїшся...не те,щоб шось запитати чи сказати..

🔹 Example #2
📈 Probability: 0.961
📝 Как бывшей студентке данного ВУЗа очень неприятно осознавать, что в общежитиях управляющий персонал каким был таким и остался- грубым "хамлом", позволяющим себе обзывать, третировать студентов и вымагать деньги!

🔹 Example #3
📈 Probability: 0.959
📝 Не раджу вступати до цього університету! Ставлення до студентів просто жахливе, грубе, нахальне. Адекватних людей одиниці, всі інші ставляться без поваги та неадекват

**2. Random prediction examples**

In [70]:
# ------------------------------------------------------------
# Random Prediction Examples (all probabilities)
# ------------------------------------------------------------

random_examples = negative_unlabeled.sample(
    n=5,
    random_state=42
)

for idx, row in random_examples.iterrows():

    print("\n" + "="*120)
    print(f"🔹 Review ID: {idx}")
    print(f"📝 {row['Review Text']}")

    print("\n📊 Probabilities:")

    for i, label in enumerate(label_cols):

        prob = row[f"{label}_prob"]

        mark = "✅" if prob >= thresholds[i] else "❌"

        print(
            f"{mark} {label:<30} "
            f"{prob:.3f}"
        )

    print("="*120)


🔹 Review ID: 1467
📝 Історичний факультет, дуже хороші викладачі, ніяких претензій до них немає, пояснюють на найвищому рівні. А до деканату іст факу є претензії та скарги. Це нормально, що у «найкращому» університеті працюють такі хамки? Жодної нормальної відповіді на запитання студента, тільки грубість. Не розуміють з першого разу та навіть не слухають, можуть собі дозволити обізвати студента. Це просто неможливо, поміняйте персонал, таке відчуття наче вони там прийшли похамили, поїли і пішли додому. Дуже щаслива, що відрахувалася та більше ніколи не вступлю у цей жахливий університет. Із приємного у цьому університеті тільки одногрупники та друзі, яких можна зустріти. Більше плюсів у цього універу немає. Якщо при вступі вам цілували руки і ноги то під час навчання об вас витиратимуть ноги тітоньки із деканату. -10000 із 10, краще б пішла в НУБІП!!

📊 Probabilities:
✅ Attitude_Towards_Students      0.859
❌ Campus_conditions              0.162
❌ Corruption                     0.331
❌ 

**3. Borderline Prediction Examples**

The following examples correspond to reviews whose predicted probabilities are closest to the decision threshold. These cases illustrate situations where the classifier is uncertain and where manual review may be beneficial.

In [71]:
for i, label in enumerate(label_cols):

    threshold = thresholds[i]

    lower = threshold - 0.03
    upper = threshold + 0.03

    borderline = negative_unlabeled[
        negative_unlabeled[f"{label}_prob"].between(
            lower,
            upper
        )
    ]

    print(
        f"\n🏷️ {label}: "
        f"{len(borderline)} borderline reviews"
    )


🏷️ Attitude_Towards_Students: 133 borderline reviews

🏷️ Campus_conditions: 76 borderline reviews

🏷️ Corruption: 90 borderline reviews

🏷️ Academic_Process_Management: 141 borderline reviews

🏷️ Education_Quality: 136 borderline reviews


In [72]:
class_name = "Corruption"

threshold = thresholds[label_cols.index(class_name)]

tmp = negative_unlabeled.copy()

tmp["distance"] = abs(
    tmp[f"{class_name}_prob"] - threshold
)

closest = tmp.sort_values(
    "distance"
).head(10)

for _, row in closest.iterrows():

    print("\n" + "="*120)

    print(
        f"📈 Probability: "
        f"{row[f'{class_name}_prob']:.3f}"
    )

    print(
        f"🎯 Threshold: "
        f"{threshold:.2f}"
    )

    print(f"📝 {row['Review Text']}")


📈 Probability: 0.501
🎯 Threshold: 0.50
📝 Не советую поступать на любой факультет этого  университета, ректор на уступки никто не идёт , даже когда он нарушил закон и признает это. Спокойно могут отчислить в любой момент , а самое интересное без  причины . Общежития ужасные , тараканы повсюду , антисанитария полная .  Если вы дорожите своим временем , прислушайтесь к моему совету !!!!

📈 Probability: 0.499
🎯 Threshold: 0.50
📝 Ужасный университет, никакого образования нет. Гребут только деньги и все. Никому не рекомендую туда поступать

📈 Probability: 0.499
🎯 Threshold: 0.50
📝 Не тратьте свои нервы и деньги!

📈 Probability: 0.502
🎯 Threshold: 0.50
📝 Що я можу сказати про УКУ – тут класна трапезна, велика бібліотека і гарний інтер’єр. Але це напевно всі плюси УКУ. За такі гроші я очікувала більше, коли вступала сюди. Викладачі викладають свої предмети посередньо, іноді взагалі халатно ставляться до своїх обов`язків. Дискримінація студентів присутня, хоча й не у відкритому вигляді. Рівень